In [1]:
from pathlib import Path
import pandas as pd
import sys

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

from src.utils.preprocess import (
    preprocess_initial_dataset,
    remove_log_features,
    split_dataset_with_val,
    train_catboost_model,
    get_feature_importance_df,
    transform_contact_region_pvz
)




pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

DATA_PATH = Path("../../data/cleaned_dataset/final_dataset_from_notebooks.csv")
RAW_DATA_PATH = Path("../../data/raw/clean_dataset.csv")
XSLX_PATH = Path("../../data/raw/PvzList_rus-2.xlsx")

In [2]:
df = pd.read_csv(DATA_PATH)
df_raw = pd.read_csv(RAW_DATA_PATH)
df = df.merge(
    df_raw[
        ["row_id", "lead_Стоимость доставки", "lead_Масса (гр)", "lead_Высота", "contact_Код ПВЗ"]
    ],
    on="row_id",
    how="left",
)
df = df.drop(columns="contact_region_pvz")
pvz_data = pd.read_excel(XSLX_PATH, sheet_name="Россия", usecols=[0, 1, 2], engine='openpyxl')
df = transform_contact_region_pvz(df, pvz_data)
df = remove_log_features(df)
del df_raw
# Приводим всё к нужному типу
df = preprocess_initial_dataset(df)
df = df.drop(columns="contact_Код ПВЗ")
df.info()

C:\Users\Михаил\AppData\Local\Temp\ipykernel_9156\1309604265.py:2: DtypeWarning: Columns (79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(RAW_DATA_PATH)


<class 'pandas.core.frame.DataFrame'>
Index: 17966 entries, 0 to 18787
Columns: 110 entries, contact_pvz_code to contact_has_pvz_code
dtypes: datetime64[ns](2), float64(18), int64(64), object(26)
memory usage: 15.2+ MB


In [27]:
print(*df.columns.tolist(), sep="\n")

contact_pvz_code
lead_weight_gm
lead_responsible_user_id
lead_utm_group
lead_utm_referrer
lead_tags
lead_utm_source
lead_Квалификация лида
is_responsible_same
has_weight
lead_utm_referrer_site
lead_has_roistat
lead_utm_id_1
lead_utm_id_2
lead_utm_id_3
lead_utm_device_type
lead_utm_site
lead_utm_position
lead_utm_reatrgeting_id
lead_utm_region_name
lead_is_utm_campaign_type_1
contact_Город
contact_region
lead_manager_category
lead_rate_is_warehouse_to_warehouse
lead_formname_has_value
lead_has_creation_date
lead_creation_date_week
lead_creation_date_month
lead_creation_date_quarter
lead_creation_date_dayofweek
lead_creation_date_sin
lead_creation_date_cos
sale_date_month
sale_date_quarter
sale_date_dayofweek
sale_date_week
sale_date_sin
sale_date_cos
lead_items_count
lead_total_quantity
lead_total_cost_from_composition
lead_has_health_supplement
lead_has_pillow
lead_has_mattress
lead_has_brace
lead_has_footwear
lead_has_accessory
lead_has_health_product
lead_categories_count
lead_has_li

Удаляем из модели субъективную оценку менеджеров

Удаляем lead_group_quality так как потенциально содержит данные из будущего

In [ ]:
df_reduced = df.drop(columns=["lead_Квалификация лида", "lead_group_quality"])

X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_reduced)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления lead_group_quality:", metrics)

In [29]:
fi = get_feature_importance_df(model, X_train, y_train)

fi.head(20)

,feature,importance
70,lead_payment_type,15.961591
5,lead_tags,4.395064
3,lead_utm_group,3.847163
14,lead_utm_device_type,3.729870
99,lead_height_bin,3.425300
91,utm_sky_brand,2.971248
71,lead_delivery_type,2.957815
6,lead_utm_source,2.833583
38,lead_items_count,2.256896
57,timedelta_between_sale_and_creation,2.149461


✅ Улучшение! Удаление признака lead_price дало PR_AUC = 0.378325

In [ ]:
df_reduced = df_reduced.drop(columns=["lead_price"])

X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_reduced)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления lead_price:", metrics)

In [31]:
fi = get_feature_importance_df(model, X_train, y_train)

fi.head(20)

,feature,importance
69,lead_payment_type,24.923440
3,lead_utm_group,7.278075
5,lead_tags,6.592564
40,lead_total_cost_from_composition,5.092226
14,lead_utm_device_type,4.743109
56,lead_source,4.514615
90,utm_sky_brand,3.148211
66,lead_group_id,2.714166
58,lead_created_ts,2.481648
35,sale_date_week,2.450030


✅ Улучшение! Удаление признака width_is_missing дало PR_AUC = 0.389970

In [5]:
df_reduced = df_reduced.drop(columns=["width_is_missing"])

X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_reduced)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления width_is_missing:", metrics)

0:	learn: 0.6375452	test: 0.6146247	best: 0.6146247 (0)	total: 212ms	remaining: 7m 3s
100:	learn: 0.7723262	test: 0.6676735	best: 0.6743555 (12)	total: 6.98s	remaining: 2m 11s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6743554887
bestIteration = 12

Shrink model to first 13 iterations.
Метрики после удаления width_is_missing: {'ROC_AUC': 0.6711080314827352, 'PR_AUC': 0.3899703955089175, 'LogLoss': 0.6843715345224443}


In [6]:
fi = get_feature_importance_df(model, X_train, y_train)

fi.head(20)

,feature,importance
68,lead_payment_type,22.579310
14,lead_utm_device_type,10.070237
13,lead_utm_id_3,9.492683
5,lead_tags,6.522557
3,lead_utm_group,4.893978
65,lead_length,4.857552
69,lead_delivery_type,3.995333
9,lead_utm_referrer_site,3.816106
58,lead_created_ts,3.029497
11,lead_utm_id_1,2.844528


In [35]:
baseline_pr_auc = 0.3899703955089175

results = []

for feature in fi["feature"]:
    print(f"\nУдаляем признак: {feature}")

    df_removed = df_reduced.drop(columns=[feature])

    X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(
        df_removed
    )

    model, metrics = train_catboost_model(
        X_train, y_train, X_val, y_val, X_test, y_test
    )

    pr_auc = metrics["PR_AUC"]

    print(f"PR_AUC: {pr_auc:.6f}")

    results.append((feature, pr_auc))

    # проверка улучшения
    if pr_auc > baseline_pr_auc:
        print(f"✅ Улучшение! Удаление признака {feature} дало PR_AUC = {pr_auc:.6f}")


Удаляем признак: lead_payment_type
0:	learn: 0.6176356	test: 0.6222230	best: 0.6222230 (0)	total: 64ms	remaining: 2m 7s
100:	learn: 0.7738426	test: 0.6521000	best: 0.6723571 (39)	total: 6.79s	remaining: 2m 7s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6723570928
bestIteration = 39

Shrink model to first 40 iterations.
PR_AUC: 0.338283

Удаляем признак: lead_utm_device_type
0:	learn: 0.6514430	test: 0.6722305	best: 0.6722305 (0)	total: 62.4ms	remaining: 2m 4s
100:	learn: 0.7745041	test: 0.6865123	best: 0.6875451 (99)	total: 6.59s	remaining: 2m 3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6875450818
bestIteration = 99

Shrink model to first 100 iterations.
PR_AUC: 0.362323

Удаляем признак: lead_utm_id_3
0:	learn: 0.6516250	test: 0.6422620	best: 0.6422620 (0)	total: 61.5ms	remaining: 2m 3s
100:	learn: 0.7708934	test: 0.6686728	best: 0.6825078 (8)	total: 6.96s	remaining: 2m 10s
Stopped by overfitting detector  (100 iterations wait)

be

In [36]:
results_df = pd.DataFrame(results, columns=["feature", "pr_auc"])
results_df = results_df.sort_values(by="pr_auc", ascending=False)

print(results_df[results_df["pr_auc"] > baseline_pr_auc])

Empty DataFrame
Columns: [feature, pr_auc]
Index: []


In [37]:
baseline_pr_auc = 0.3899703955089175

for n in range(90, 29, -1):
    col_to_drop = fi["feature"].tail(n).to_list()

    df_test = df_reduced.drop(columns=col_to_drop)

    X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_test)

    model, metrics = train_catboost_model(
        X_train, y_train, X_val, y_val, X_test, y_test
    )

    pr_auc = metrics["PR_AUC"]

    print(f"tail({n}) -> PR_AUC: {pr_auc:.6f}")

    if pr_auc > baseline_pr_auc:
        print("\nIMPROVEMENT FOUND")
        print(f"tail({n}) gives PR_AUC = {pr_auc:.6f} (baseline = {baseline_pr_auc})")
        break

0:	learn: 0.6409768	test: 0.6493899	best: 0.6493899 (0)	total: 49ms	remaining: 1m 37s
100:	learn: 0.7224334	test: 0.6692222	best: 0.6816220 (4)	total: 4.6s	remaining: 1m 26s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6816219588
bestIteration = 4

Shrink model to first 5 iterations.
tail(90) -> PR_AUC: 0.355564
0:	learn: 0.6550814	test: 0.6530321	best: 0.6530321 (0)	total: 45.2ms	remaining: 1m 30s
100:	learn: 0.7202827	test: 0.6727969	best: 0.6744498 (93)	total: 4.71s	remaining: 1m 28s
200:	learn: 0.7660357	test: 0.6529815	best: 0.6756504 (108)	total: 9.65s	remaining: 1m 26s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.675650373
bestIteration = 108

Shrink model to first 109 iterations.
tail(89) -> PR_AUC: 0.353945
0:	learn: 0.6465424	test: 0.6644386	best: 0.6644386 (0)	total: 43.9ms	remaining: 1m 27s
100:	learn: 0.7237571	test: 0.6835573	best: 0.6877731 (1)	total: 4.57s	remaining: 1m 25s
Stopped by overfitting detector  (100 iterations 

IMPROVEMENT FOUND\
tail(87) gives PR_AUC = 0.406579 (baseline = 0.3899703955089175)

In [ ]:
col_to_drop = fi["feature"].tail(87).to_list()

df_reduced = df_reduced.drop(columns=col_to_drop)
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(
    df_reduced
)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления 87 неважных признаков:", metrics)

0:	learn: 0.6250918	test: 0.6609869	best: 0.6609869 (0)	total: 48ms	remaining: 1m 35s
100:	learn: 0.7319893	test: 0.7006587	best: 0.7015039 (99)	total: 4.81s	remaining: 1m 30s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7015038802
bestIteration = 99

Shrink model to first 100 iterations.
Метрики после удаления 87 неважных признаков: {'ROC_AUC': 0.6761906525050778, 'PR_AUC': 0.40657857277830695, 'LogLoss': 0.6960768336903173}


In [39]:
X_train.columns.to_list()

['lead_utm_group',
 'lead_utm_referrer',
 'lead_tags',
 'has_weight',
 'lead_utm_referrer_site',
 'lead_utm_id_1',
 'lead_utm_id_3',
 'lead_utm_device_type',
 'lead_creation_date_quarter',
 'lead_created_ts',
 'lead_length',
 'lead_payment_type',
 'lead_delivery_type',
 'lead_utm_campaign_missing',
 'lead_group_missing',
 'lead_height_bin',
 'delivery_cost_missing']

In [12]:
import joblib
joblib.dump(model, "../../model/model.pkl")

['../../model/model.pkl']

In [8]:
df_reduced.to_csv(str(PROJECT_ROOT / "bench" / "correct_dataset.csv"))

In [ ]:
# correct_file = str(PROJECT_ROOT / "bench" / "correct_dataset.csv")
# incorrect_file = str(PROJECT_ROOT / "bench" / "incorrect_dataset.csv")

# # Чтение файлов
# try:
#     correct_df = pd.read_csv(correct_file)
#     incorrect_df = pd.read_csv(incorrect_file)
# except FileNotFoundError as e:
#     print(f"Ошибка: Файл не найден — {e}")
#     exit()
# except Exception as e:
#     print(f"Ошибка при чтении файлов: {e}")
#     exit()

# # Проверка, что датафреймы имеют одинаковую структуру (количество строк и столбцов)
# if correct_df.shape != incorrect_df.shape:
#     print("Предупреждение: Форматы датафреймов отличаются. Сравнение может быть некорректным.")

# # Список для хранения имён столбцов с несовпадениями
# columns_with_discrepancies = []

# # Перебор столбцов и сравнение значений
# for column in correct_df.columns:
#     # Проверяем, есть ли столбец в обоих датафреймах
#     if column not in incorrect_df.columns:
#         columns_with_discrepancies.append(column)
#         continue

#     # Сравниваем значения в столбце, игнорируя NaN (NaN != NaN)
#     comparison = correct_df[column].equals(incorrect_df[column])
#     if not comparison:
#         # Более детальная проверка: ищем конкретные несовпадения
#         mask = (correct_df[column] != incorrect_df[column]) & \
#                 (~correct_df[column].isna()) & \
#                 (~incorrect_df[column].isna())
#         if mask.any() or len(correct_df[column]) != len(incorrect_df[column]):
#             columns_with_discrepancies.append(column)

# # Вывод результата
# if columns_with_discrepancies:
#     print("Столбцы с несовпадениями:")
#     for col in columns_with_discrepancies:
#         print(f"- {col}")
# else:
#     print("Все столбцы совпадают.")


Столбцы с несовпадениями:
- contact_region_pvz
- lead_utm_device_type
- sale_hour
- sale_ts
- contact_to_lead_hours
